In [1]:
import pandas as pd
import os
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# 1. Setup Paths and Load Data
RAW_DIR = "../data/raw/"
PROCESSED_DIR = "../data/processed/"

print("Loading communications data...")
df_comms = pd.read_parquet(os.path.join(RAW_DIR, "all_communications.parquet"))

# 2. Initialize the NLP Analyzer
analyzer = SentimentIntensityAnalyzer()

# 3. Define Extraction Functions
def extract_sentiment(text):
    """Returns a compound score from -1 (Extremely Negative) to 1 (Extremely Positive)"""
    if pd.isna(text): 
        return 0.0
    return analyzer.polarity_scores(str(text))['compound']

def extract_urgency(text):
    """Binary flag indicating if urgency keywords are present"""
    if pd.isna(text): 
        return 0
    urgency_keywords = ['urgent', 'asap', 'immediate', 'critical', 'escalate', 'risk', 'delayed', 'fail']
    text_lower = str(text).lower()
    return 1 if any(word in text_lower for word in urgency_keywords) else 0

print("Running NLP extraction on 151k+ emails (This may take a minute)...")

# 4. Apply Functions to the Email Bodies
df_comms['nlp_sentiment_score'] = df_comms['body'].apply(extract_sentiment)
df_comms['nlp_urgency_flag'] = df_comms['body'].apply(extract_urgency)

# 5. Aggregate NLP Features to the Customer Level
# Since a customer might send multiple emails, we want their *average* sentiment and *max* urgency.
nlp_features = df_comms.groupby('customer_id').agg(
    avg_sentiment=('nlp_sentiment_score', 'mean'),
    min_sentiment=('nlp_sentiment_score', 'min'), # Captures their angriest moment
    total_urgent_emails=('nlp_urgency_flag', 'sum'),
    total_emails_sent=('message_id', 'count')
).reset_index()

print("NLP Feature Extraction Complete.")
print(nlp_features.head())

# Save the features for the ML model
nlp_features.to_parquet(os.path.join(PROCESSED_DIR, "nlp_customer_features.parquet"), index=False)

Loading communications data...
Running NLP extraction on 151k+ emails (This may take a minute)...
NLP Feature Extraction Complete.
  customer_id  avg_sentiment  min_sentiment  total_urgent_emails  \
0  0001114321       0.547109         0.4927                    0   
1  0002992449       0.189541         0.0000                    0   
2  0003062521       0.532600         0.4927                    0   
3  0003264409       0.380400         0.0258                    0   
4  0004078317       0.182837         0.0000                    0   

   total_emails_sent  
0                 11  
1                 39  
2                  9  
3                 15  
4                 19  
